# M1.2 · Gradient descent varyantları

Tam batch, stokastik ve mini-batch gradient descent ile momentum. Bir kuadratikte ve üç noktalık regresyonda adımları elle atıyor, karıştırılmış mini-batch'leri saf Python'da üretiyor, NumPy'da epoch/batch döngüsünü kuruyor, PyTorch'ta `DataLoader` + `SGD(momentum)` ile aynı yörüngeyi doğruluyor ve en sonda 48 farklı (lr, β) çiftini tek tensörle eğitiyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m1-2-gd-variants.html](https://m-gonul.github.io/pytorch-lab/m1-2-gd-variants.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m1_2_gd_variants.py
CHECKS = {
 "b": "import random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def mb():\n        f = need(ns, \"minibatches\")\n        bs = f(103, 10, random.Random(1))\n        ensure(len(bs) == 11 and all(len(b) == 10 for b in bs[:-1]) and len(bs[-1]) == 3, f\"batch boyları {[len(b) for b in bs]}\")\n        flat = [i for b in bs for i in b]\n        ensure(sorted(flat) == list(range(103)), \"her örnek tam bir kez görülmeli\")\n        ensure(flat != list(range(103)), \"sıra karıştırılmamış\")\n        ensure(f(20, 5, random.Random(7)) == f(20, 5, random.Random(7)), \"aynı rng tohumu aynı batch'leri vermeli\")\n    R.test(\"minibatches: karıştırılmış, örtüşmeyen, eksiksiz\", mb)\n\n    xs, ys = [1.0, 2.0, 3.0], [2.0, 4.0, 5.0]\n\n    def full():\n        w, b = need(ns, \"sgd_epoch\")(xs, ys, 0.0, 0.0, 0.1, 3, random.Random(0))\n        close([w, b], [5 / 3, 11 / 15], what=\"batch_size = n → tek GD adımı\")\n    R.test(\"batch_size = n: M1.1'deki tam GD adımı\", full)\n\n    def conv():\n        rng = random.Random(0)\n        w = b = 0.0\n        for _ in range(3000):\n            w, b = need(ns, \"sgd_epoch\")(xs, ys, w, b, 0.01, 1, rng)\n        close([w, b], [1.5, 2 / 3], atol=0.05, rtol=0, what=\"SGD (batch 1) sonucu\")\n        return f\"SGD 3000 epoch: w = {w:.3f}, b = {b:.3f} (kapalı form 1.5, 0.667; gürültü yüzünden tam oturmaz)\"\n    R.test(\"batch_size = 1 kapalı forma yaklaşıyor\", conv)\n\n    def mom():\n        p = need(ns, \"momentum_path\")(lambda w: 2 * (w - 3), 0.0, 0.25, 0.5, 4)\n        ensure(len(p) == 4, \"4 değer bekleniyordu\")\n        close(p[:2], [1.5, 3.0])\n        p0 = need(ns, \"momentum_path\")(lambda w: 2 * (w - 3), 0.0, 0.25, 0.0, 2)\n        close(p0, [1.5, 2.25], what=\"β = 0 düz GD olmalı\")\n    R.test(\"momentum_path: [1.5, 3.0, …], β = 0 düz GD\", mom)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"train: en fazla 2 katman döngü\", lambda: max_loops(src, 2, func=\"train\", why=\"örnekler üzerinde döngü olmamalı\"))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(3)\n    X = rng.normal(size=(400, 3))\n    y = X @ np.array([1.0, -2.0, 0.5]) + 1.0 + 0.05 * rng.normal(size=400)\n\n    def full_batch():\n        w, b, L = need(ns, \"train\")(X, y, lr=0.1, batch_size=400, epochs=1)\n        e = -y\n        close(w, -0.1 * 2 * X.T @ e / 400, what=\"tam batch tek adım w\")\n        close(b, -0.1 * 2 * e.mean(), what=\"tam batch tek adım b\")\n    R.test(\"batch_size = n, beta = 0: tek tam GD adımı\", full_batch)\n\n    def conv():\n        w, b, L = need(ns, \"train\")(X, y, lr=0.05, batch_size=16, epochs=40, beta=0.0)\n        L = np.asarray(L)\n        ensure(L.shape == (40,), f\"losses shape {L.shape}\")\n        Xt = np.column_stack([X, np.ones(400)])\n        th = np.linalg.lstsq(Xt, y, rcond=None)[0]\n        close(w, th[:3], atol=0.03, rtol=0, what=\"w\"); close(b, th[3], atol=0.03, rtol=0, what=\"b\")\n        return f\"son epoch kaybı {L[-1]:.5f}\"\n    R.test(\"mini-batch SGD en küçük kareler çözümüne yaklaşıyor\", conv)\n\n    def det():\n        a = need(ns, \"train\")(X, y, lr=0.05, batch_size=32, epochs=3, seed=5)\n        b_ = need(ns, \"train\")(X, y, lr=0.05, batch_size=32, epochs=3, seed=5)\n        close(a[0], b_[0], what=\"aynı seed\")\n    R.test(\"seed ile tekrarlanabilir\", det)\n\n    def ill():\n        Xi = rng.normal(size=(400, 2)) * np.array([1.0, 10.0])\n        yi = Xi @ np.array([3.0, 0.2])\n        _, _, L0 = need(ns, \"train\")(Xi, yi, lr=0.004, batch_size=400, epochs=60, beta=0.0)\n        _, _, L9 = need(ns, \"train\")(Xi, yi, lr=0.004, batch_size=400, epochs=60, beta=0.9)\n        ensure(L9[-1] < 0.1 * L0[-1], f\"β = 0.9: {L9[-1]:.4g}, β = 0: {L0[-1]:.4g} — momentum belirgin daha iyi olmalı\")\n        return f\"60 adım sonra kayıp: β = 0 → {L0[-1]:.4g}, β = 0.9 → {L9[-1]:.4g}\"\n    R.test(\"kötü koşullu problemde momentum hızlandırıyor\", ill)\n",
 "d": "import torch\nfrom torch import nn\nfrom labcheck import close, need, ensure, uses_any\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"DataLoader ve optim.SGD kullanılıyor\", lambda: ensure(uses_any(src, [\"DataLoader\"]) and uses_any(src, [\"SGD\"]), \"DataLoader + torch.optim.SGD bekleniyordu\"))\n\n    def path():\n        p = need(ns, \"momentum_path_torch\")(0.0, 0.25, 0.5, 5)\n        w, v, ref = 0.0, 0.0, []\n        for _ in range(5):\n            v = 0.5 * v + 2 * (w - 3); w -= 0.25 * v; ref.append(w)\n        close(p, ref, rtol=1e-6)\n        return \"PyTorch: \" + \", \".join(f\"{t:.4f}\" for t in p)\n    R.test(\"SGD(momentum) yörüngesi elle formülle aynı\", path)\n\n    def tr():\n        torch.manual_seed(0)\n        X = torch.randn(600, 4)\n        y = X @ torch.tensor([1.0, -2.0, 0.5, 0.0]) + 1.0 + 0.05 * torch.randn(600)\n        m = need(ns, \"train_torch\")(X, y, lr=0.02, batch_size=32, epochs=40, momentum=0.9)\n        ensure(isinstance(m, nn.Linear), \"nn.Linear döndürmeli\")\n        Xt = torch.cat([X, torch.ones(600, 1)], 1)\n        th = torch.linalg.lstsq(Xt, y.unsqueeze(1)).solution.squeeze()\n        close(m.weight.detach().view(-1), th[:4], atol=0.03, rtol=0, what=\"weight\")\n        close(m.bias.detach(), th[4:], atol=0.03, rtol=0, what=\"bias\")\n    R.test(\"train_torch en küçük kareler çözümüne ulaşıyor\", tr)\n",
 "e": "import torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef one(X, y, lr, beta, steps):\n    w = torch.zeros(X.shape[1], dtype=X.dtype); v = torch.zeros_like(w)\n    for _ in range(steps):\n        g = 2 * X.T @ (X @ w - y) / X.shape[0]\n        v = beta * v + g\n        w = w - lr * v\n    return ((X @ w - y) ** 2).mean()\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"sweep: yalnız adım döngüsü\", lambda: max_loops(src, 1, func=\"sweep\", why=\"kombinasyonlar üzerinde döngü olmamalı\"))\n    g = torch.Generator().manual_seed(0)\n    X = (torch.randn(200, 2, generator=g) * torch.tensor([1.0, 8.0])).double()\n    y = X @ torch.tensor([2.0, 0.3], dtype=torch.float64)\n    lr_g, b_g = torch.meshgrid(torch.logspace(-4, -2.2, 6, dtype=torch.float64), torch.linspace(0, 0.95, 8, dtype=torch.float64), indexing=\"ij\")\n    lrs, betas = lr_g.reshape(-1), b_g.reshape(-1)\n\n    def eq():\n        L = need(ns, \"sweep\")(X, y, lrs, betas, 60)\n        ensure(tuple(L.shape) == (48,), f\"shape {tuple(L.shape)}\")\n        ref = torch.stack([one(X, y, float(a), float(b), 60) for a, b in zip(lrs, betas)])\n        close(torch.nan_to_num(L, nan=1e30, posinf=1e30), torch.nan_to_num(ref, nan=1e30, posinf=1e30), rtol=1e-6, atol=1e-9)\n        k = int(torch.nan_to_num(L, nan=float(\"inf\")).argmin())\n        return f\"en iyi: lr = {float(lrs[k]):.2e}, β = {float(betas[k]):.2f}, kayıp {float(L[k]):.3g}\"\n    R.test(\"48 kombinasyon tek tek eğitimle aynı\", eq)\n    if \"sweep\" not in ns:\n        return\n    Xf, yf = X.float(), y.float()\n    t_loop = bench(lambda: [one(Xf, yf, float(a), float(b), 100) for a, b in zip(lrs, betas)], repeat=3, number=1)\n    t_vec = bench(lambda: ns[\"sweep\"](Xf, yf, lrs.float(), betas.float(), 100), repeat=3, number=1)\n    L = ns[\"sweep\"](Xf, yf, lrs.float(), betas.float(), 100).view(6, 8)\n    rows = [[f\"{float(lr_g[i, 0]):.1e}\"] + [(\"ıraksadı\" if not torch.isfinite(L[i, j]) or L[i, j] > 1e6 else f\"{float(L[i, j]):.2g}\") for j in range(0, 8, 2)] for i in range(6)]\n    R.table(\"100 adım sonra kayıp (satır: lr, sütun: β)\", [\"lr \\\\ β\"] + [f\"{float(b_g[0, j]):.2f}\" for j in range(0, 8, 2)], rows,\n            note=\"Momentum, kararlı kalan en büyük lr'yi düşürür ama o sınırın altında çok daha hızlı yakınsar.\")\n    R.table(\"48 (lr, β) çifti × 100 adım\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for: 48 ayrı eğitim\", fmt_time(t_loop), \"1×\"], [\"senin sweep\", fmt_time(t_vec), f\"{t_loop / t_vec:,.0f}× hızlı\"]])\n"
}

In [ ]:
import importlib, labcheck, checks_m1_2_gd_variants as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

M1.1'deki gradient descent her adımda **tüm** veriyi kullanıyordu: $g=\nabla L(\theta)=\frac1n\sum_i\nabla\ell_i(\theta)$. $n$ milyonlarca olunca tek adım bile pahalı. Üç seçenek:

| yöntem | adımda kullanılan | adım maliyeti | gürültü |
|---|---|---|---|
| batch GD | $n$ örnek | $O(n)$ | yok |
| SGD | 1 örnek | $O(1)$ | yüksek |
| mini-batch | $B$ örnek (32–4096) | $O(B)$ | $\propto 1/B$ |

Bir epoch, verinin tamamının bir kez görülmesi: $\lceil n/B\rceil$ adım. Pratikte "SGD" denince karıştırılmış mini-batch kastedilir.

Momentum gradyanların üstel hareketli ortalamasını tutar:

$$
v_t=\beta v_{t-1}+g_t,\qquad \theta_{t+1}=\theta_t-\eta\,v_t .
$$

Bu, PyTorch'un `SGD(momentum=β)` formülüdür (dampening = 0). Tutarlı yönlerde hız birikir, salınan yönler sönümlenir.

## 1 · Elle

<!-- çözüm -->

Önce tek boyutlu kuadratikte düz GD ile momentumu yan yana koy. Düz GD minimuma her adımda kalan mesafenin yarısı kadar yaklaşıyor ($w=0\to1.5\to2.25\to\dots$). Momentum ikinci adımda biriktirdiği hızla tam $3$'e ulaşıyor. Bu şanslı bir seçim; daha büyük $\beta$ ile hedefi aşar ve salınır.

Sonra regresyon verisinde tek örnekli ve iki örnekli gradyanları hesapla. M1.1'deki tam batch gradyanı $\partial_w=-16.67$ idi; SGD ($-4$) ve mini-batch ($-23$) aynı **yönü** gösteriyor ama büyüklükleri farklı. Ortalamaları alınırsa tam gradyana yaklaşırlar.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Kuadratik $L(w)=(w-3)^2$, $\nabla L=2(w-3)$, $w_0=0$, $\eta=0.25$. Düz GD**

- `g0`: $g_0=\nabla L(w_0)$
- `w1`: $w_1=w_0-\eta g_0$
- `w2`: $w_2$

**Momentum, $\beta=0.5$: $v_t=\beta v_{t-1}+g_t$, $w_{t+1}=w_t-\eta v_t$, $v_{-1}=0$**

- `mv1`: $v_1$ (ikinci adımın hızı)
- `mw2`: $w_2$

**Regresyon $x=[1,2,3]$, $y=[2,4,5]$, $w=b=0$, $\eta=0.1$, MSE**

- `sgd_w`: SGD, yalnız 1. örnek: yeni $w$
- `sgd_b`: … yeni $b$
- `mb_gw`: Mini-batch {2., 3. örnek}: $\partial L/\partial w$
- `mb_gb`: … $\partial L/\partial b$
- `steps`: $n=1000$, batch 32: epoch başına adım

In [ ]:
cevaplar = dict(
    g0=None,
    w1=None,
    w2=None,
    mv1=None,
    mw2=None,
    sgd_w=None,
    sgd_b=None,
    mb_gw=None,
    mb_gb=None,
    steps=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"g0":[-6,0],"w1":[1.5,0.001],"w2":[2.25,0.001],"mv1":[-6,0.001],"mw2":[3,0.001],"sgd_w":[0.4,0.001],"sgd_b":[0.4,0.001],"mb_gw":[-23,0.001],"mb_gb":[-9,0.001],"steps":[32,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, tek özellikli regresyon ($\hat y=wx+b$, MSE). `random` serbest, `numpy`/`torch` yasak.

- `minibatches(n, batch_size, rng)` — `range(n)`'i `rng.shuffle` ile karıştır, `batch_size`'lık parçalara böl, **liste listesi** döndür. Son parça küçük olabilir.
- `sgd_epoch(xs, ys, w, b, lr, batch_size, rng)` → `(w, b)`; her mini-batch'in **ortalama** gradyanıyla bir adım.
- `momentum_path(grad, w0, lr, beta, steps)` → `[w_1, …, w_steps]`; tek değişkenli, `grad` bir fonksiyon.

Kontrol: `batch_size = n` ile tek GD adımı, `batch_size = 1` ile kapalı forma yakınsama, momentum yörüngesi elle bulduğunla aynı.

In [ ]:
%%writefile stage_b.py
import random


def minibatches(n, batch_size, rng):
    ...


def sgd_epoch(xs, ys, w, b, lr, batch_size, rng):
    ...


def momentum_path(grad, w0, lr, beta, steps):
    ...


print(minibatches(10, 4, random.Random(0)))
print(momentum_path(lambda w: 2 * (w - 3), 0.0, 0.25, 0.5, 4))   # [1.5, 3.0, …]

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy'da çok özellikli regresyon. Eğitim döngüsü iki katmanlı: epoch'lar ve her epoch'taki batch'ler. Örnekler **üzerinde** döngü yok (kontrol `train` içinde en fazla 2 katman döngüye izin verir).

- `train(X, y, lr, batch_size, epochs, beta=0.0, seed=0)` → `(w, b, losses)`. Her epoch başında `rng = np.random.default_rng(seed)` ile oluşturduğun üreteçten `rng.permutation(n)`. `losses[k]`: $k$. epoch **sonundaki** tam veri MSE'si. Momentum `w` ve `b` için ayrı hız vektörleriyle.

Kontrol, kötü koşullu bir problemde (`X`'in sütun ölçekleri 1 ve 10) aynı lr ile $\beta=0.9$'un $\beta=0$'dan belirgin şekilde daha düşük kayba indiğini doğrular.

In [ ]:
%%writefile stage_c.py
import numpy as np


def train(X, y, lr, batch_size, epochs, beta=0.0, seed=0):
    """→ (w, b, losses: (epochs,)). En fazla 2 katman döngü (epoch, batch)."""
    ...


rng = np.random.default_rng(0)
X = rng.normal(size=(500, 3))
y = X @ np.array([1.0, -2.0, 0.5]) + 1.0 + 0.1 * rng.normal(size=500)
w, b, losses = train(X, y, lr=0.05, batch_size=32, epochs=30, beta=0.9)
print(w, b, losses[-1])

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch'ta mini-batch'leri `torch.utils.data.DataLoader` üretir:

```python
loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)
for xb, yb in loader:   # bir epoch
    ...
```

- `momentum_path_torch(w0, lr, beta, steps)` — $L(w)=(w-3)^2$'yi `torch.optim.SGD(momentum=beta)` ile minimize et, her adımdan sonraki $w$'leri liste olarak döndür. Elle bulduğun $[1.5, 3.0, …]$ çıkmalı; bu, PyTorch'un momentum formülünün seninkiyle aynı olduğunun kanıtı.
- `train_torch(X, y, lr=0.05, batch_size=32, epochs=50, momentum=0.9)` → eğitilmiş `nn.Linear`.

In [ ]:
%%writefile stage_d.py
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def momentum_path_torch(w0, lr, beta, steps):
    """L(w) = (w − 3)² için SGD(momentum=beta) yörüngesi (liste)."""
    ...


def train_torch(X, y, lr=0.05, batch_size=32, epochs=50, momentum=0.9):
    ...


print(momentum_path_torch(0.0, 0.25, 0.5, 4))    # [1.5, 3.0, …]

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Hangi (lr, β) çifti? 6 lr × 8 β = 48 kombinasyon. M1.1'deki gibi hepsini **tek tensör** olarak eğit: ağırlıklar `W: (P, d)`, hızlar `V: (P, d)`, her kombinasyonun kendi `lr` ve `beta`'sı `(P,)` vektörlerinde. Tam batch (gürültüsüz), böylece sonuç deterministik.

- `sweep(X, y, lrs, betas, steps)` → son kayıplar `(P,)`; `lrs` ve `betas` aynı uzunlukta `(P,)` (ızgara önceden düzleştirilmiş). Yalnız adım döngüsü; kombinasyonlar üzerinde döngü yok. Bias yok (veri ortalanmış).

Kontrol sonucu tek tek eğitilmiş modellerle karşılaştırır, en iyi çifti ve hız kazancını raporlar.

In [ ]:
%%writefile stage_e.py
import torch


def sweep(X, y, lrs, betas, steps):
    """P = len(lrs) model; tam batch momentum GD. → son MSE kayıpları (P,). Yalnız adım döngüsü."""
    P, d = lrs.shape[0], X.shape[1]
    W = torch.zeros(P, d, dtype=X.dtype)
    V = torch.zeros(P, d, dtype=X.dtype)
    ...


torch.manual_seed(0)
X = torch.randn(300, 2) * torch.tensor([1.0, 8.0])
y = X @ torch.tensor([2.0, 0.3])
lr_grid, beta_grid = torch.meshgrid(torch.logspace(-4, -2, 6), torch.linspace(0, 0.95, 8), indexing="ij")
losses = sweep(X, y, lr_grid.reshape(-1), beta_grid.reshape(-1), 100)
print(losses.view(6, 8))

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Robbins & Monro (1951) ve Polyak & Juditsky (1992): stokastik yaklaşım ve ortalama

[Robbins & Monro — A Stochastic Approximation Method (1951); Polyak & Juditsky — Acceleration of Stochastic Approximation by Averaging (SIAM J. Control 1992)](https://projecteuclid.org/journals/annals-of-mathematical-statistics/volume-22/issue-3/A-Stochastic-Approximation-Method/10.1214/aoms/1177729586.full)

Gürültülü doğrusal regresyonda SGD'yi adım takvimleri ηₜ = c/tᵅ (α ∈ {0.3, 0.5, 0.7, 1}) ile çalıştır; Robbins–Monro koşullarını (Σηₜ = ∞, Σηₜ² < ∞) sağlayan ve sağlamayanların yakınsama davranışını karşılaştır. Polyak–Ruppert ortalamasını (iteratların ortalaması) ekle: α = 0.5 ile ortalanmış SGD'nin α = 1'in (en iyi c ile) asimptotik hızına, c seçimine duyarsız şekilde ulaştığını göster.


**Adımlar**

1. Hatanın t ile log-log eğimini ölç (beklenen: ortalamalıda −1).

**Kabul ölçütleri**

- Koşulları sağlamayan sabit adımın gürültü tabanında kaldığı grafik.
- Ortalama alma ile c'ye duyarsız O(1/t) hata.

## Kaynaklar

- [Stanford CS229 — Lecture notes: LMS, batch vs stochastic gradient descent](https://cs229.stanford.edu/main_notes.pdf) — doğrulanmadı
- [Dive into Deep Learning §12.4–12.6 SGD, Minibatch SGD, Momentum](https://d2l.ai/chapter_optimization/index.html) — CC BY-SA 4.0 / MIT-0
- [Goh — Why Momentum Really Works (Distill)](https://distill.pub/2017/momentum/) — CC BY 4.0
- [PyTorch — torch.optim.SGD (momentum formülü)](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html) — BSD-3